# MindeesAI — Google Colab GPU training notebook

Trains the `home-11gb` variant (~280M params) on a Colab T4 GPU. Use Colab when Kaggle quota is exhausted or you want a burst session.

## Before you run

1. **Runtime → Change runtime type:**
   - **Hardware accelerator: T4 GPU** (or any GPU available on free tier)
   - **Runtime shape: High-RAM** (if you have access to it)
2. **🔑 sidebar icon → Add new secret:**
   - **Name**: `HF_TOKEN`
   - **Value**: your HuggingFace **write-scoped** token from https://huggingface.co/settings/tokens
   - Toggle on **Notebook access**

## Colab gotchas

- **90-minute idle disconnect** — keep the tab visible, use a keep-alive extension, or run during off-peak hours
- **No real cron** — must trigger this notebook manually each time
- **GPU not guaranteed on free tier** — if Cell 1 reports CPU only, restart the runtime and try again later
- **12h session cap** — practical limit is often shorter due to disconnects; use `--ckpt-every 1000` to minimize lost work

## What this notebook does

Same flow as the Kaggle notebook, scoped to `colab-burst` revision so multiple training environments stay isolated:

1. Clone `aashir-athar/mindeesai` repo
2. Pull `HF_TOKEN` from Colab Secrets
3. Resume-fetch from HF revision `colab-burst`
4. Train `home-11gb` for 200k steps OR until disconnect
5. Push back to HF revision `colab-burst` (independent from `main`, `small-weekly`, `kaggle-weekly`)

## Cell 1 — Clone repo + install dependencies

In [ ]:
!rm -rf /content/mindeesai
!git clone -q https://github.com/aashir-athar/mindeesai.git /content/mindeesai
%cd /content/mindeesai
!pip install -q -r scripts/train/requirements.txt
!pip install -q huggingface_hub

import torch
print(f"PyTorch {torch.__version__} · CUDA available: {torch.cuda.is_available()} · GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
if not torch.cuda.is_available():
    print("\n⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU, then restart the runtime.")

## Cell 2 — Load `HF_TOKEN` from Colab Secrets

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

assert os.environ["HF_TOKEN"].startswith("hf_"), "HF_TOKEN does not look like a valid HuggingFace token — check the 🔑 sidebar"
print(f"HF_TOKEN loaded (length {len(os.environ['HF_TOKEN'])} chars)")

## Cell 3 — Resume from HF `colab-burst` revision

First run 404s on every file and we train from random init. Subsequent runs pick up where the previous session left off — even if Colab disconnected mid-training, the last `--ckpt-every 1000` save survives on HF.

In [ ]:
import os
from huggingface_hub import hf_hub_download

REPO = "aashir-athar/mindeesai-base"
REVISION = "colab-burst"   # change to "kaggle-weekly" to continue Kaggle's run, or "main" for local RTX work

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("tokenizer", exist_ok=True)

for filename, local_dir in [
    ("base.bin",        "checkpoints"),
    ("torch-resume.pt", "checkpoints"),
    ("tokenizer.json",  "tokenizer"),
]:
    try:
        p = hf_hub_download(repo_id=REPO, filename=filename, revision=REVISION,
                            local_dir=local_dir, local_dir_use_symlinks=False,
                            token=os.environ["HF_TOKEN"])
        print(f"  ✓ {filename}: {os.path.getsize(p):,} bytes → {p}")
    except Exception as e:
        msg = str(e).lower()
        if "404" in msg or "not found" in msg or "entry not found" in msg:
            print(f"  · {filename}: not on HF yet (first run, will train fresh)")
        else:
            print(f"  ✗ {filename}: {e}")

## Cell 3.5 — Train BPE tokenizer if missing

On the first run nothing was on HF, so the resume fetch above 404'd on `tokenizer.json`. Train one fresh from `scripts/data/corpus.txt` at `vocab=50000` — this writes to `tokenizer/tokenizer.json` which Cell 4 (and the upload in Cell 5) then use.

On subsequent runs the resume fetch already pulled the saved tokenizer from HF, so this cell is a no-op (preserves token-ID stability across sessions — critical for resume correctness).

In [ ]:
import os, subprocess

tok_path = "tokenizer/tokenizer.json"
if not os.path.exists(tok_path) or os.path.getsize(tok_path) == 0:
    print("tokenizer.json not present — training fresh BPE at vocab=50000")
    subprocess.run([
        "python", "scripts/train/tokenizer_train.py",
        "--corpus", "scripts/data/corpus.txt",
        "--vocab-size", "50000",
        "--out", tok_path,
    ], check=True)
    print(f"✓ tokenizer trained → {tok_path} ({os.path.getsize(tok_path):,} bytes)")
else:
    print(f"✓ tokenizer/tokenizer.json present ({os.path.getsize(tok_path):,} bytes) — skipping fresh BPE training")

## Cell 4 — Train

Default: `home-11gb` (~280M params), batch 4 × grad-accum 2 = effective batch 8, fp16 + grad-ckpt, full `mix-broadbrain.json` recipe.

Colab T4 step time is similar to Kaggle T4. Expect ~100-200k steps in a full session, less if you get disconnected.

**If you hit OOM:** drop `--batch` to 2 or 1. **If GPU class is weak (K80):** lower `--ckpt-every` to 500 so disconnect costs less.

In [ ]:
!python scripts/train/pretrain.py \
    --variant home-11gb \
    --corpus scripts/data/corpus.txt \
    --tokenizer tokenizer/tokenizer.json \
    --steps 200000 \
    --batch 4 \
    --grad-accum 2 \
    --lr 3e-4 \
    --warmup 1000 \
    --wd 0.1 \
    --val-frac 0.05 \
    --val-every 500 \
    --ckpt-every 1000 \
    --base-weight 1.0 \
    --distill-weight 4.0 \
    --completion-only-loss 1 \
    --persona-loss-weight 0.05 \
    --mix-config scripts/data/mix-broadbrain.json \
    --resume checkpoints/torch-resume.pt \
    --amp \
    --grad-ckpt \
    --log data/training-metrics.jsonl \
    --out checkpoints/base.bin \
    --torch-ckpt checkpoints/torch-resume.pt

## Cell 5 — Push trained checkpoint back to HF `colab-burst`

**Always pushes to `colab-burst` revision only.** Your `main`, `small-weekly`, and `kaggle-weekly` revisions are untouched.

If you want this Colab run as production, set `HF_MODEL_REVISION=colab-burst` on Vercel — never modify `main` from a Colab notebook.

In [ ]:
import os
from huggingface_hub import HfApi, create_branch

REPO = "aashir-athar/mindeesai-base"
TARGET_REVISION = "colab-burst"   # ⚠️ NEVER change to "main" by accident

api = HfApi(token=os.environ["HF_TOKEN"])

# Make sure the branch exists (idempotent)
try:
    create_branch(REPO, branch=TARGET_REVISION, exist_ok=True, token=os.environ["HF_TOKEN"])
except Exception as e:
    print(f"create_branch note: {e}")

uploaded = 0
for local, in_repo in [
    ("checkpoints/base.bin",         "base.bin"),
    ("checkpoints/torch-resume.pt",  "torch-resume.pt"),
    ("tokenizer/tokenizer.json",     "tokenizer.json"),
    ("data/training-metrics.jsonl",  "training-metrics.jsonl"),
]:
    if not os.path.exists(local):
        print(f"  · {local}: not found, skipping")
        continue
    try:
        api.upload_file(
            path_or_fileobj=local,
            path_in_repo=in_repo,
            repo_id=REPO,
            revision=TARGET_REVISION,
            commit_message=f"Colab session — {in_repo}",
        )
        print(f"  ✓ {local} → {REPO}@{TARGET_REVISION}:{in_repo}")
        uploaded += 1
    except Exception as e:
        print(f"  ✗ {local}: {e}")

print(f"\nUploaded {uploaded} files. View on HF: https://huggingface.co/{REPO}/tree/{TARGET_REVISION}")
print("To use this in production: set HF_MODEL_REVISION=colab-burst on Vercel.")

## Done

Your checkpoint is now on HF revision `colab-burst`. Close the tab and the runtime will release. Next time you open this notebook, Cell 3 pulls the latest checkpoint and Cell 4 resumes training.

**If you got disconnected mid-Cell 4:** the `--ckpt-every 1000` saves mean the last ~1000 steps' worth of progress is preserved on disk. Re-run Cell 5 to push that partial progress to HF, then re-run the notebook. Resume will pick up from where you left off.